# Checkpoint 2: Research Question Formation

**Project:** Toxicity Detection in News Comments  
**Dataset:** Civil Comments (Jigsaw/TensorFlow Datasets)  
Civil Comments (news comment dataset) – TensorFlow Datasets

Link: https://www.tensorflow.org/datasets/catalog/civil_comments

## 1. Project Scope and Dataset Recap

### Dataset Overview
- **Name:** Civil Comments (Jigsaw Toxic Comments dataset)
- **Source:** TensorFlow Datasets / Kaggle Jigsaw competitions
- **Size:** ~1.8 million comments from online news articles
- **Structure:** Each row is one comment with text, toxicity labels, metadata (user IDs, article IDs, timestamps)

### Key Features
- `comment_text`: The raw comment text
- `target` (or `toxicity`): Overall toxicity score (0–1, continuous)
- Additional labels: `severe_toxicity`, `insult`, `identity_attack`, etc.
- Metadata: `created_date`, `article_id`, engagement counts

### Course Techniques
- **Text preprocessing:** Tokenization, stopword removal, basic cleaning
- **Bag-of-Words / TF–IDF:** Classic text vectorization for baseline models
- **Logistic Regression / SVM:** Simple linear classifiers for toxicity prediction
- **Evaluation metrics:** Accuracy, precision, recall, F1, ROC-AUC

### Beyond-Course Techniques
- **Transformer-based embeddings:** Using pretrained models like BERT, DistilBERT, or RoBERTa for rich text representations
- **Topic modeling:** LDA or neural topic models to discover thematic patterns in comments
- **Advanced imbalance handling:** SMOTE, focal loss, or threshold tuning for rare toxic cases
- **Deep learning architectures:** Fine-tuning transformers or using attention mechanisms

### EDA Findings Summary

From Checkpoint 1 exploratory data analysis, I identified several key patterns:

Observation 1.
Most comments are short to medium length, but there is a long tail of very long comments.

Hypothesis.
Very short and very long comments may behave differently (e.g., short “throwaway” toxic replies vs longer explanations), so using more flexible text representations (like embeddings) may capture patterns better than simple fixed-length bag‑of‑words.

Observation 2.
Only about 8% of comments are labeled toxic (with a 0.5 threshold), so the data is very imbalanced toward non‑toxic comments.

Hypothesis.
Naive models will look good on accuracy but ignore many toxic cases; I may need class‑balanced training, different thresholds, or better metrics (like F1 or ROC‑AUC) to properly evaluate models.

## 2. Research Question Definition

Based on EDA findings, I propose three research questions

### RQ1: How does severe class imbalance affect toxicity classification performance, and which rebalancing strategies most effectively improve detection of toxic comments?

**Data Mining Task Type:** Supervised Text Classification with Class Imbalance Handling

**Relevant Algorithms:**
- **Course:** TF-IDF + Logistic Regression (baseline without rebalancing)
- **Course:** Class-weighted Logistic Regression
- **External:** SMOTE (Synthetic Minority Oversampling Technique)
- **External:** Threshold tuning with ROC curve analysis

**Evaluation Criteria:**
- Precision, Recall, F1-score for toxic class
- ROC-AUC
- Precision-Recall curve analysis
- Confusion matrix visualization

**Justification:** EDA revealed 8% toxic / 92% non-toxic imbalance. Naive models achieve 92% accuracy by predicting all non-toxic but fail at content moderation's core purpose.

---

### RQ2: Do transformer-based text embeddings improve detection of subtle toxic comments compared to bag-of-words representations?

**Data Mining Task Type:** Text Representation Learning + Classification

**Relevant Algorithms:**
- **Course:** TF-IDF vectorization + SVM classifier
- **External:** Pretrained DistilBERT embeddings + fine-tuning
- **External:** Contextual word embeddings (transformers)

**Evaluation Criteria:**
- Overall F1, accuracy, ROC-AUC
- Performance on "borderline" comments (toxicity score 0.3–0.7)
- Error analysis on mis-classified examples
- Representation quality visualization (t-SNE/UMAP)

**Justification:** EDA showed comment length variation (median 35 words, long tail >300). Bag-of-words loses context needed for sarcasm, implied insults, coded language.

---

### RQ3: How do discussion topics relate to toxicity levels, and can topic features improve classification?

**Data Mining Task Type:** Unsupervised Topic Discovery + Feature Engineering

**Relevant Algorithms:**
- **Course:** TF-IDF + Logistic Regression (baseline without topics)
- **External:** LDA (Latent Dirichlet Allocation) for topic modeling
- **External:** Topic-enhanced classification (topic proportions as features)

**Evaluation Criteria:**
- Topic coherence scores
- Average toxicity per topic
- Classification improvement (ΔF1, ΔAUC) when adding topic features
- Interpretability of discovered topics

**Justification:** EDA showed temporal variation suggesting topic shifts. News comments span politics, sports, culture—certain topics may be high-risk, enabling targeted moderation.

##  RQ-to-Method Mapping Table

| Research Question | Data Mining Task | Course Techniques | External Techniques | Key Metrics |
|-------------------|------------------|-------------------|---------------------|-------------|
| **RQ1: Class Imbalance** | Supervised Classification | TF-IDF + LR<br>Class-weighted LR | SMOTE<br>Threshold tuning | Precision, Recall, F1<br>ROC-AUC<br>PR curves |
| **RQ2: Representation Quality** | Text Representation + Classification | TF-IDF + SVM | DistilBERT embeddings<br>Fine-tuned transformers | F1, Accuracy<br>Performance on borderline cases<br>t-SNE visualization |
| **RQ3: Topics & Toxicity** | Topic Discovery + Feature Engineering | TF-IDF + LR baseline | LDA topic modeling<br>Topic-enhanced features | Topic coherence<br>Avg. toxicity per topic<br>ΔF1, ΔAUC |


## 3. Motivation and Feasibility Analysis

### Motivation

**Why these questions matter:**
1. **EDA-driven:** Each RQ directly addresses specific patterns found in EDA:
   - RQ1 motivated by 8% toxic / 92% non-toxic imbalance
   - RQ2 motivated by comment length variation (short vs long tail)
   - RQ3 motivated by temporal variation suggesting topic shifts

2. **Practical impact:** Content moderation systems must balance recall (catching toxic content) with precision (avoiding false flags). These RQs address real deployment challenges.

3. **Non-triviality:** Course techniques alone (TF-IDF + simple classifiers) miss important patterns:
   - Ignore class imbalance → poor minority class performance
   - Ignore word order/context → miss subtle toxicity
   - Ignore topic structure → miss thematic risk patterns

### Feasibility

**RQ1 (Class Imbalance):**
-  **Learnable:** SMOTE is well-documented in `imbalanced-learn` library
- **Implementable:** Threshold tuning requires only probability predictions (available from all classifiers)
-  **Risk:** SMOTE on high-dimensional TF-IDF may be slow; will sample data or use reduced dimensions
-  **Verified:** Additional EDA confirmed we have ~144K toxic examples (sufficient for rebalancing tests)

**RQ2 (Transformers):**
-  **Learnable:** HuggingFace `transformers` library has extensive tutorials
-  **Implementable:** DistilBERT is lightweight (66M parameters vs BERT's 110M)
- **Risk:** GPU memory limits; will use Colab GPU, batch processing, and sample ~100K comments for fine-tuning
-  **Verified:** Successfully loaded DistilBERT in method testing (see below)

**RQ3 (Topic Modeling):**
- **Learnable:** LDA implemented in `gensim` and `sklearn` with good documentation
- **Implementable:** LDA scales well to large text corpora
-  **Risk:** Topic interpretability is subjective; will use coherence metrics and manual inspection
- **Verified:** Can run LDA on sample data in reasonable time

### Computational Risks and Mitigation
- **Risk:** Full dataset (1.8M comments) too large for some methods
- **Mitigation:** Strategic sampling (e.g., 200K for baselines, 100K for transformers) while maintaining class balance
- **Parameter sensitivity:** Will use grid search for critical hyperparameters; document all choices

## 4. Methodological Planning

### Implementation Plan

**RQ1: Class Imbalance**
1. **Course baselines:** TF-IDF (max_features=5000) + Logistic Regression (default, then class_weight='balanced')
2. **External methods:** Apply SMOTE to training data; implement threshold tuning via ROC curve
3. **Evaluation:** Stratified 80/20 train/test split; report per-class metrics + PR curves
4. **Baselines for comparison:** "Always predict majority class" and "Random guess"

**RQ2: Text Representation**
1. **Course baseline:** TF-IDF (max_features=10000) + SVM (kernel='linear')
2. **External methods:** DistilBERT embeddings → fine-tune final layers for toxicity classification
3. **Evaluation:** Overall metrics + focused analysis on borderline comments (0.3 ≤ toxicity ≤ 0.7)
4. **Baselines for comparison:** TF-IDF + SVM vs DistilBERT frozen embeddings vs fine-tuned

**RQ3: Topic Modeling**
1. **Course baseline:** TF-IDF + Logistic Regression (no topic features)
2. **External methods:** LDA (num_topics=10-20, test range) → extract topic proportions → augment features
3. **Evaluation:** Topic coherence scores; toxicity by topic; classification Δ metrics
4. **Baselines for comparison:** Text-only model vs topic-enhanced model

### Method and Metric Plan Summary

| Method Category | Specific Techniques | Key Hyperparameters | Success Metrics |
|-----------------|---------------------|---------------------|-----------------|
| **Preprocessing** | Lowercase, remove URLs/emails, tokenization | max_features (5K–10K) | - |
| **Course Text Mining** | TF-IDF, BoW, Logistic Regression, SVM | C (regularization), kernel | F1, ROC-AUC |
| **Imbalance Handling** | Class weights, SMOTE, threshold tuning | sampling_strategy, threshold | Toxic class recall, precision |
| **Deep Embeddings** | DistilBERT, fine-tuning | learning_rate, epochs, batch_size | F1 on borderline cases, overall AUC |
| **Topic Modeling** | LDA | num_topics (10–20), alpha, beta | Coherence score, ΔF1 |

On my honor, I declare the following resources:


(1) Collaborators  
-(None.)

(2) Web Sources  
- Gensim / scikit-learn topic modeling docs – LDA usage and topic coherence guidelines
- Scikit-learn User Guide – model training, evaluation metrics, and imbalance handling
- Python pandas and matplotlib documentation – data inspection, simple EDA plots, and table formatting
-- Kaggle – “Jigsaw Unintended Bias in Toxicity Classification” dataset: https://www.kaggle.com/c/jigsaw-unintended-bias-in-toxicity-classification/data
- Borkan, D., Dixon, L., Sorensen, J., Thain, N., & Vasserman, N. (2019). Nuanced metrics for measuring unintended bias with real data. In *Proceedings of the World Wide Web Conference (WWW ’19)*.  


(3) AI Tools  
- Perplexity : used to brainstorm research question angles, refine wording of the three RQs, map each RQ to appropriate course/external methods, and sanity‑check feasibility and risk mitigation text.Help with the formatting of the text and table mapping.